# Apartment Rent Prediction (CRISP-DM)

**BMDS2003 Data Science Project — Group 4**

Regression: predict monthly apartment rent (USD). <br>Dataset: UCI Apartment for Rent Classified (100K). <br>Models: Linear Regression (baseline), Decision Tree, Random Forest, Hist Gradient Boosting. Each model section includes three diagnostic graphs. Metrics: MAE, RMSE, R2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

## 1. Data Understanding

In [ ]:
df = pd.read_csv('apartments_for_rent_classified_100K.csv', sep=';', encoding='cp1252', low_memory=False)
df.shape

In [ ]:
df.head()

In [ ]:
df[['price', 'square_feet', 'bathrooms', 'bedrooms']].describe()

### Exploratory Data Analysis

The following graphs explore the raw data before preparation and motivate the cleaning decisions.

In [ ]:
eda_df = df[df['price_type'] == 'Monthly'].copy()
for c in ['price', 'square_feet', 'bathrooms', 'bedrooms', 'latitude', 'longitude']:
    eda_df[c] = pd.to_numeric(eda_df[c], errors='coerce')
p99 = eda_df['price'].quantile(0.99)
plt.figure(figsize=(7.5, 5))
plt.hist(eda_df['price'], bins=60, range=(0, p99), color='#2ca02c', alpha=0.85)
plt.axvline(eda_df['price'].median(), color='k', ls='--', lw=1.3, label=f"median ${eda_df['price'].median():.0f}")
plt.xlabel('Monthly rent ($)'); plt.ylabel('Count')
plt.title('Distribution of Monthly Rent (right-skewed)'); plt.legend()
plt.tight_layout(); plt.savefig('eda_price_hist.png', dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 5))
ax[0].boxplot(eda_df['price'].dropna()); ax[0].set_title('Rent - outliers present'); ax[0].set_ylabel('$')
ax[1].boxplot(eda_df['square_feet'].dropna()); ax[1].set_title('Square feet - outliers present'); ax[1].set_ylabel('sqft')
plt.suptitle('Boxplots before outlier removal (justify IQR filtering)')
plt.tight_layout(); plt.savefig('eda_boxplots.png', dpi=140); plt.show()

In [ ]:
num = eda_df[['price', 'square_feet', 'bathrooms', 'bedrooms', 'latitude', 'longitude']].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(num, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num.columns, rotation=40, ha='right')
ax.set_yticks(range(len(num))); ax.set_yticklabels(num.columns)
for i in range(len(num)):
    for j in range(len(num)):
        ax.text(j, i, f"{num.iloc[i, j]:.2f}", ha='center', va='center', color='black', fontsize=9)
plt.colorbar(im, fraction=0.046); ax.set_title('Correlation Heatmap (numeric features)')
plt.tight_layout(); plt.savefig('eda_correlation.png', dpi=140); plt.show()

In [ ]:
sample = eda_df[(eda_df['price'] < p99) & (eda_df['square_feet'] < eda_df['square_feet'].quantile(0.99))].sample(6000, random_state=42)
plt.figure(figsize=(7.5, 5.5))
plt.scatter(sample['square_feet'], sample['price'], s=6, alpha=0.25, color='#1f77b4', edgecolors='none')
plt.xlabel('Square feet'); plt.ylabel('Monthly rent ($)')
plt.title('Rent vs Square Feet (positive, non-linear)')
plt.tight_layout(); plt.savefig('eda_price_vs_sqft.png', dpi=140); plt.show()

In [ ]:
top_states = eda_df['state'].value_counts().head(15).index
ms = eda_df[eda_df['state'].isin(top_states)].groupby('state')['price'].mean().sort_values(ascending=False)
plt.figure(figsize=(8.5, 5))
plt.bar(ms.index, ms.values, color='#ff7f0e')
plt.xlabel('State'); plt.ylabel('Average monthly rent ($)')
plt.title('Average Rent by State (top 15 by listing count)')
plt.tight_layout(); plt.savefig('eda_rent_by_state.png', dpi=140); plt.show()

## 2. Data Preparation

In [ ]:
dataframe_clean = df[df['price_type'] == 'Monthly'].copy()
dataframe_clean = dataframe_clean.drop(columns=['id', 'category', 'title', 'body', 'currency', 'price_display', 'price_type', 'address', 'source', 'time', 'cityname'])
dataframe_clean.shape

In [ ]:
def iqr_bounds(series, k):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - k * IQR, Q3 + k * IQR

price_low, price_high = iqr_bounds(dataframe_clean['price'], 3.0)
sqft_low, sqft_high = iqr_bounds(dataframe_clean['square_feet'], 3.0)
dataframe_clean = dataframe_clean[(dataframe_clean['price'] >= price_low) & (dataframe_clean['price'] <= price_high) & (dataframe_clean['square_feet'] >= sqft_low) & (dataframe_clean['square_feet'] <= sqft_high)]
dataframe_clean.shape

In [ ]:
dataframe_clean['amenities'] = dataframe_clean['amenities'].fillna('none')
dataframe_clean['pets_allowed'] = dataframe_clean['pets_allowed'].fillna('none')
dataframe_clean['bathrooms'] = dataframe_clean['bathrooms'].fillna(dataframe_clean['bathrooms'].median())
dataframe_clean['bedrooms'] = dataframe_clean['bedrooms'].fillna(dataframe_clean['bedrooms'].median())
dataframe_clean = dataframe_clean.dropna(subset=['state', 'latitude', 'longitude'])
dataframe_clean.shape

In [ ]:
dataframe_clean['amenity_count'] = dataframe_clean['amenities'].apply(lambda x: 0 if x == 'none' else len(x.split(',')))
dataframe_clean['pets_flag'] = dataframe_clean['pets_allowed'].apply(lambda x: 0 if x == 'none' else 1)

In [ ]:
df_model = dataframe_clean.drop(columns=['amenities', 'pets_allowed'])
df_model = pd.get_dummies(df_model, columns=['fee', 'has_photo', 'state'], drop_first=True)
df_model.shape

In [ ]:
X = df_model.drop(columns=['price'])
y = df_model['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
results = []

def evaluate(name, model):
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"{name}: MAE {mae:.2f} | RMSE {rmse:.2f} | R2 {r2:.4f}")

def top_importance(model):
    imp = pd.Series(model.feature_importances_, index=X_train.columns)
    state_cols = [c for c in X_train.columns if c.startswith('state_')]
    agg = imp[[c for c in X_train.columns if not c.startswith('state_')]].copy()
    agg['state (location)'] = imp[state_cols].sum()
    return agg.sort_values(ascending=False).head(8).sort_values()

## 3. Modelling

### Model 1 — Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
evaluate('Linear Regression', lr)

In [ ]:
pred = lr.predict(X_test)
plt.figure(figsize=(6.2, 6.2))
plt.scatter(y_test, pred, s=5, alpha=0.2, color='#d62728', edgecolors='none')
lo, hi = y_test.min(), y_test.max()
plt.plot([lo, hi], [lo, hi], 'k--', lw=1.3)
plt.xlabel('Actual rent ($)'); plt.ylabel('Predicted rent ($)')
plt.title('LR - Predicted vs Actual'); plt.tight_layout(); plt.show()

In [ ]:
residual = y_test.values - lr.predict(X_test)
plt.figure(figsize=(7, 5))
plt.scatter(lr.predict(X_test), residual, s=5, alpha=0.2, color='#d62728', edgecolors='none')
plt.axhline(0, color='k', lw=1.2, ls='--')
plt.xlabel('Predicted rent ($)'); plt.ylabel('Residual ($)')
plt.title('LR - Residuals vs Predicted'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(residual, bins=60, color='#d62728', alpha=0.85)
plt.axvline(0, color='k', lw=1.2, ls='--')
plt.xlabel('Residual ($)'); plt.ylabel('Count')
plt.title('LR - Residual Distribution'); plt.tight_layout(); plt.show()

### Model 2 — Decision Tree

In [ ]:
dt_final = DecisionTreeRegressor(max_depth=12, random_state=42)
dt_final.fit(X_train, y_train)
evaluate('Decision Tree (depth=12)', dt_final)

In [ ]:
depths = [3, 6, 9, 12, 15, 20]
train_r2, test_r2 = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    train_r2.append(r2_score(y_train, m.predict(X_train)))
    test_r2.append(r2_score(y_test, m.predict(X_test)))
plt.figure(figsize=(7, 5))
plt.plot(depths, train_r2, 'o-', label='Train R2')
plt.plot(depths, test_r2, 's-', label='Test R2')
plt.axvline(12, color='gray', ls=':')
plt.xlabel('max_depth'); plt.ylabel('R2')
plt.title('Decision Tree - Validation Curve'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
t = top_importance(dt_final)
plt.figure(figsize=(7.5, 5))
plt.barh(t.index, t.values, color='#ff7f0e')
plt.xlabel('Importance'); plt.title('Decision Tree - Feature Importance')
plt.tight_layout(); plt.show()

In [ ]:
pdt = dt_final.predict(X_test)
tiers = pd.cut(y_test, [0, 1000, 1500, 2000, 2500, 1e9], labels=['<1k', '1-1.5k', '1.5-2k', '2-2.5k', '2.5k+'])
mae_by_tier = pd.Series(np.abs(y_test.values - pdt)).groupby(tiers.values).mean()
plt.figure(figsize=(7, 5))
plt.bar(mae_by_tier.index.astype(str), mae_by_tier.values, color='#ff7f0e')
plt.xlabel('Actual rent tier'); plt.ylabel('MAE ($)')
plt.title('Decision Tree - MAE by Rent Tier'); plt.tight_layout(); plt.show()

### Model 3 — Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate('Random Forest', rf)

In [ ]:
t = top_importance(rf)
plt.figure(figsize=(7.5, 5))
plt.barh(t.index, t.values, color='#1f77b4')
plt.xlabel('Importance'); plt.title('Random Forest - Feature Importance')
plt.tight_layout(); plt.show()

In [ ]:
rres = y_test.values - rf.predict(X_test)
bd = X_test['bedrooms'].clip(upper=4).astype(int).values
groups = [rres[bd == b] for b in [0, 1, 2, 3, 4]]
plt.figure(figsize=(7, 5))
plt.boxplot(groups, labels=['0', '1', '2', '3', '4+'], showfliers=False)
plt.axhline(0, color='r', lw=1, ls='--')
plt.xlabel('Bedrooms'); plt.ylabel('Residual ($)')
plt.title('Random Forest - Residuals by Bedrooms'); plt.tight_layout(); plt.show()

In [ ]:
prf = rf.predict(X_test)
plt.figure(figsize=(6.8, 6))
hbn = plt.hexbin(y_test, prf, gridsize=45, cmap='Blues', mincnt=1)
plt.plot([lo, hi], [lo, hi], 'r--', lw=1.3)
plt.colorbar(hbn, label='count')
plt.xlabel('Actual rent ($)'); plt.ylabel('Predicted rent ($)')
plt.title('Random Forest - Predicted vs Actual (density)'); plt.tight_layout(); plt.show()

### Model 4 — Hist Gradient Boosting

Standard Gradient Boosting is slow on 100K rows, so we use `HistGradientBoostingRegressor` (histogram-based, built for large data).

In [ ]:
hgb = HistGradientBoostingRegressor(random_state=42)
hgb.fit(X_train, y_train)
evaluate('Hist Gradient Boosting', hgb)

In [ ]:
n = len(X_train)
sizes = [int(f * n) for f in [0.2, 0.4, 0.6, 0.8, 1.0]]
ltrain, ltest = [], []
for k in sizes:
    mm = HistGradientBoostingRegressor(random_state=42).fit(X_train.iloc[:k], y_train.iloc[:k])
    ltrain.append(r2_score(y_train.iloc[:k], mm.predict(X_train.iloc[:k])))
    ltest.append(r2_score(y_test, mm.predict(X_test)))
plt.figure(figsize=(7, 5))
plt.plot(sizes, ltrain, 'o-', label='Train R2')
plt.plot(sizes, ltest, 's-', label='Test R2')
plt.xlabel('Training samples'); plt.ylabel('R2')
plt.title('Hist Gradient Boosting - Learning Curve'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
hres = y_test.values - hgb.predict(X_test)
plt.figure(figsize=(7, 5))
plt.scatter(hgb.predict(X_test), hres, s=5, alpha=0.2, color='#2ca02c', edgecolors='none')
plt.axhline(0, color='k', lw=1.2, ls='--')
plt.xlabel('Predicted rent ($)'); plt.ylabel('Residual ($)')
plt.title('Hist GB - Residuals vs Predicted'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(np.abs(hres), bins=60, color='#2ca02c', alpha=0.85)
plt.xlabel('Absolute error ($)'); plt.ylabel('Count')
plt.title('Hist GB - Absolute Error Distribution'); plt.tight_layout(); plt.show()

## 4. Evaluation

In [ ]:
results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.bar(results_df['Model'], results_df['R2'], color=['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728'])
ax.set_ylabel('Test R2'); ax.set_title('Model Comparison - 100K (Test R2)'); ax.set_ylim(0, 0.95)
for i, v in enumerate(results_df['R2']):
    ax.text(i, v + 0.012, f"{v:.3f}", ha='center', fontweight='bold')
plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.savefig('model_comparison_r2.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. Export Model for Deployment

Random Forest is the most accurate, but its saved model is ~140 MB. The Streamlit app deploys the compact Hist Gradient Boosting model (~0.4 MB, R2 approx 0.77).

In [ ]:
joblib.dump(hgb, 'rent_model.joblib')
joblib.dump(list(X_train.columns), 'model_columns.joblib')
geo = dataframe_clean.groupby('state')[['latitude', 'longitude']].median()
joblib.dump(geo, 'state_geo.joblib')